# AI Video Summarizer — Colab Training

**Pipeline:**
1. Mount Google Drive & check GPU
2. Install dependencies
3. Write project source files
4. Download MSR-VTT dataset from Kaggle
5. Build caption dataset (ResNet18 feature extraction)
6. Train the captioning model (BiGRU + Bahdanau attention + LSTM decoder)
7. Save the trained model back to Google Drive

> **Runtime:** Make sure to select `Runtime → Change runtime type → GPU (T4 or better)`

## 1. Mount Google Drive & Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install Dependencies

In [ ]:
%%capture
!pip install kaggle nltk tqdm opencv-python-headless pillow

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
print("Dependencies ready.")

## 3. Set Up Project Directory

In [ ]:
import os

PROJECT = '/content/ai-video-summarizer'
DRIVE_SAVE = '/content/drive/MyDrive/ai-video-summarizer-outputs'

for d in [
    f'{PROJECT}/src/data',
    f'{PROJECT}/src/models',
    f'{PROJECT}/src/training',
    f'{PROJECT}/src/utils',
    f'{PROJECT}/data/raw/msvtt',
    f'{PROJECT}/data/processed/msvtt',
    f'{PROJECT}/outputs/models',
    DRIVE_SAVE,
]:
    os.makedirs(d, exist_ok=True)

# Write __init__.py files so imports work
for pkg in ['src', 'src/data', 'src/models', 'src/training', 'src/utils']:
    open(f'{PROJECT}/{pkg}/__init__.py', 'a').close()

os.chdir(PROJECT)
print(f"Working directory: {os.getcwd()}")

## 4. Write Source Files

In [ ]:
# ── vocabulary.py ──────────────────────────────────────────────────────────────
vocabulary_py = '''
from collections import Counter

SPECIAL_TOKENS = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}


class Vocabulary:
    def __init__(self):
        self.word2idx = dict(SPECIAL_TOKENS)
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.freq: Counter = Counter()

    def build(self, captions, min_freq=2):
        for cap in captions:
            for word in cap.lower().split():
                self.freq[word] += 1
        for word, count in self.freq.items():
            if count >= min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, caption, max_len=30):
        tokens = ["<sos>"] + caption.lower().split()[: max_len - 2] + ["<eos>"]
        ids = [self.word2idx.get(t, self.word2idx["<unk>"]) for t in tokens]
        ids += [0] * (max_len - len(ids))
        return ids[:max_len]

    def decode(self, ids):
        words = []
        for i in ids:
            word = self.idx2word.get(i, "<unk>")
            if word == "<eos>":
                break
            if word not in ("<pad>", "<sos>", "<unk>"):
                words.append(word)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)
'''
with open(f'{PROJECT}/src/data/vocabulary.py', 'w') as f:
    f.write(vocabulary_py)
print("vocabulary.py written")

In [ ]:
# ── caption_model.py ───────────────────────────────────────────────────────────
caption_model_py = '''
import torch
import torch.nn as nn
import torch.nn.functional as F


class VideoEncoder(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=512, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, features):
        outputs, hidden = self.gru(features)
        outputs = self.dropout(outputs)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        hidden = torch.tanh(self.fc(hidden))
        return outputs, hidden


class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)

    def forward(self, encoder_outputs, decoder_hidden):
        enc_att = self.encoder_att(encoder_outputs)
        dec_att = self.decoder_att(decoder_hidden).unsqueeze(1)
        scores = self.full_att(torch.tanh(enc_att + dec_att)).squeeze(2)
        weights = F.softmax(scores, dim=1)
        context = (encoder_outputs * weights.unsqueeze(2)).sum(dim=1)
        return context, weights


class CaptionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, encoder_dim, hidden_dim, attention_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(encoder_dim, hidden_dim, attention_dim)
        self.gru = nn.GRU(
            input_size=embed_dim + encoder_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(token))
        context, weights = self.attention(encoder_outputs, hidden.squeeze(0))
        gru_input = torch.cat([embedded, context], dim=1).unsqueeze(1)
        output, hidden = self.gru(gru_input, hidden)
        prediction = self.fc_out(self.dropout(output.squeeze(1)))
        return prediction, hidden, weights


class VideoCaptionModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=256,
        encoder_hidden=512,
        decoder_hidden=512,
        attention_dim=256,
        input_dim=512,
        encoder_layers=2,
        dropout=0.3,
    ):
        super().__init__()
        self.encoder = VideoEncoder(
            input_dim=input_dim,
            hidden_dim=encoder_hidden,
            num_layers=encoder_layers,
            dropout=dropout,
        )
        self.decoder = CaptionDecoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            encoder_dim=encoder_hidden * 2,
            hidden_dim=decoder_hidden,
            attention_dim=attention_dim,
            dropout=dropout,
        )

    def forward(self, features, captions, teacher_forcing_ratio=0.5):
        batch_size = features.size(0)
        max_len = captions.size(1)
        vocab_size = self.decoder.fc_out.out_features

        encoder_outputs, hidden = self.encoder(features)
        hidden = hidden.unsqueeze(0)

        outputs = torch.zeros(batch_size, max_len, vocab_size, device=features.device)
        input_token = captions[:, 0]

        for t in range(1, max_len):
            pred, hidden, _ = self.decoder(input_token, hidden, encoder_outputs)
            outputs[:, t] = pred
            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            input_token = captions[:, t] if use_teacher else pred.argmax(1)

        return outputs
'''
with open(f'{PROJECT}/src/models/caption_model.py', 'w') as f:
    f.write(caption_model_py)
print("caption_model.py written")

In [ ]:
# ── build_caption_dataset.py ───────────────────────────────────────────────────
build_dataset_py = '''
import os
import json
import pickle
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

import torch
import torchvision.models as models
import torchvision.transforms as transforms

from src.data.vocabulary import Vocabulary

MSVTT_VIDEO_DIR = "data/raw/msvtt/TrainValVideo"
MSVTT_ANNO_FILE = "data/raw/msvtt/train_val_videodatainfo.json"
OUTPUT_DIR = "data/processed/msvtt"
VOCAB_PATH = "outputs/models/caption_vocab.pkl"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEGMENT_SECONDS = 2
MAX_CAPTION_LEN = 30
MIN_WORD_FREQ = 2


def load_resnet():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = torch.nn.Identity()
    model.eval()
    model.to(DEVICE)
    return model


def get_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def extract_video_features(video_path, resnet, transform, segment_seconds=2):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        cap.release()
        return None

    frame_interval = max(1, int(fps * segment_seconds))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    features = []

    segment_idx = 0
    while True:
        start_frame = segment_idx * frame_interval
        if start_frame >= total_frames:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        ret, frame = cap.read()
        if not ret:
            break
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        tensor = transform(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feat = resnet(tensor).squeeze().cpu().numpy()
        features.append(feat)
        segment_idx += 1

    cap.release()
    return np.array(features, dtype=np.float32) if features else None


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(VOCAB_PATH), exist_ok=True)

    print("Loading MSR-VTT annotations...")
    with open(MSVTT_ANNO_FILE, "r") as f:
        data = json.load(f)

    video_captions = {}
    for sentence in data["sentences"]:
        vid = sentence["video_id"]
        cap = sentence["caption"].strip()
        video_captions.setdefault(vid, []).append(cap)

    print(f"Total videos in annotations: {len(video_captions)}")

    all_captions = [c for caps in video_captions.values() for c in caps]
    vocab = Vocabulary()
    vocab.build(all_captions, min_freq=MIN_WORD_FREQ)
    print(f"Vocabulary size: {len(vocab)}")

    with open(VOCAB_PATH, "wb") as f:
        pickle.dump(vocab, f)
    print(f"Vocab saved -> {VOCAB_PATH}")

    resnet = load_resnet()
    transform = get_transform()

    dataset = []
    skipped = 0

    for video_id in tqdm(sorted(video_captions.keys()), desc="Processing videos"):
        video_path = os.path.join(MSVTT_VIDEO_DIR, f"{video_id}.mp4")
        if not os.path.exists(video_path):
            skipped += 1
            continue

        features = extract_video_features(video_path, resnet, transform)
        if features is None or len(features) == 0:
            skipped += 1
            continue

        for caption in video_captions[video_id]:
            encoded = vocab.encode(caption, max_len=MAX_CAPTION_LEN)
            dataset.append({
                "video_id": video_id,
                "features": features,
                "caption": encoded,
                "caption_text": caption,
            })

    print(f"Total samples: {len(dataset)}  |  Videos skipped: {skipped}")

    save_path = os.path.join(OUTPUT_DIR, "dataset.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(dataset, f)
    print(f"Dataset saved -> {save_path}")


if __name__ == "__main__":
    main()
'''
with open(f'{PROJECT}/src/data/build_caption_dataset.py', 'w') as f:
    f.write(build_dataset_py)
print("build_caption_dataset.py written")

In [ ]:
# ── train_caption.py ───────────────────────────────────────────────────────────
train_caption_py = '''
import math
import os
import pickle
import random
import sys

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from src.models.caption_model import VideoCaptionModel
from src.data.vocabulary import Vocabulary  # noqa — needed for pickle

DATASET_PATH = "data/processed/msvtt/dataset.pkl"
VOCAB_PATH = "outputs/models/caption_vocab.pkl"
MODEL_OUT = "outputs/models/caption_model.pt"

EMBED_DIM = 256
ENCODER_HIDDEN = 512
DECODER_HIDDEN = 512
ATTENTION_DIM = 256
INPUT_DIM = 512
ENCODER_LAYERS = 2
DROPOUT = 0.3

BATCH_SIZE = 128
EPOCHS = 30
LR = 1e-3
TEACHER_FORCING = 0.5
MAX_SEQ_LEN = 60
VAL_SPLIT = 0.1
CLIP_GRAD = 1.0
RANDOM_SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class CaptionDataset(Dataset):
    def __init__(self, samples, max_seq_len=60):
        self.samples = samples
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        features = torch.tensor(s["features"], dtype=torch.float32)
        if len(features) > self.max_seq_len:
            features = features[: self.max_seq_len]
        caption = torch.tensor(s["caption"], dtype=torch.long)
        return features, caption, len(features)


def collate_fn(batch):
    features, captions, lengths = zip(*batch)
    max_len = max(lengths)
    feat_dim = features[0].shape[-1]
    padded = torch.zeros(len(features), max_len, feat_dim)
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded[i, :l] = f
    return padded, torch.stack(captions), torch.tensor(lengths, dtype=torch.long)


def train_epoch(model, loader, optimizer, criterion, teacher_forcing):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for features, captions, _ in loader:
        features, captions = features.to(DEVICE), captions.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(features, captions, teacher_forcing_ratio=teacher_forcing)
        output_flat = outputs[:, 1:].reshape(-1, outputs.size(-1))
        target_flat = captions[:, 1:].reshape(-1)
        loss = criterion(output_flat, target_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()
        non_pad = (target_flat != 0).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    return total_loss / max(total_tokens, 1)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for features, captions, _ in loader:
            features, captions = features.to(DEVICE), captions.to(DEVICE)
            outputs = model(features, captions, teacher_forcing_ratio=0.0)
            output_flat = outputs[:, 1:].reshape(-1, outputs.size(-1))
            target_flat = captions[:, 1:].reshape(-1)
            loss = criterion(output_flat, target_flat)
            non_pad = (target_flat != 0).sum().item()
            total_loss += loss.item() * non_pad
            total_tokens += non_pad
    return total_loss / max(total_tokens, 1)


def compute_bleu(model, loader, vocab):
    try:
        from nltk.translate.bleu_score import corpus_bleu
    except ImportError:
        return 0.0
    model.eval()
    references, hypotheses = [], []
    with torch.no_grad():
        for features, captions, _ in loader:
            features, captions = features.to(DEVICE), captions.to(DEVICE)
            outputs = model(features, captions, teacher_forcing_ratio=0.0)
            preds = outputs.argmax(dim=-1)
            for i in range(len(captions)):
                ref = vocab.decode(captions[i].cpu().tolist()).split()
                hyp = vocab.decode(preds[i].cpu().tolist()).split()
                references.append([ref])
                hypotheses.append(hyp)
    return corpus_bleu(references, hypotheses)


def main():
    random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    print("Loading dataset...")
    with open(DATASET_PATH, "rb") as f:
        dataset = pickle.load(f)
    print(f"  Samples: {len(dataset)}")

    with open(VOCAB_PATH, "rb") as f:
        vocab = pickle.load(f)
    print(f"  Vocab size: {len(vocab)}")

    random.shuffle(dataset)
    split = int(len(dataset) * (1 - VAL_SPLIT))
    train_set = CaptionDataset(dataset[:split], MAX_SEQ_LEN)
    val_set = CaptionDataset(dataset[split:], MAX_SEQ_LEN)
    print(f"  Train: {len(train_set)}  |  Val: {len(val_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

    model = VideoCaptionModel(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        encoder_hidden=ENCODER_HIDDEN,
        decoder_hidden=DECODER_HIDDEN,
        attention_dim=ATTENTION_DIM,
        input_dim=INPUT_DIM,
        encoder_layers=ENCODER_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Device: {DEVICE}\n")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)
    best_val_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, TEACHER_FORCING)
        val_loss = eval_epoch(model, val_loader, criterion)
        bleu = compute_bleu(model, val_loader, vocab)
        scheduler.step(val_loss)

        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f} | "
            f"PPL {math.exp(val_loss):.1f} | "
            f"BLEU-4 {bleu:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "vocab_size": len(vocab),
                    "config": {
                        "embed_dim": EMBED_DIM,
                        "encoder_hidden": ENCODER_HIDDEN,
                        "decoder_hidden": DECODER_HIDDEN,
                        "attention_dim": ATTENTION_DIM,
                        "input_dim": INPUT_DIM,
                        "encoder_layers": ENCODER_LAYERS,
                        "dropout": DROPOUT,
                    },
                },
                MODEL_OUT,
            )
            print(f"           -> best model saved (val={val_loss:.4f})")

    print(f"\nDone. Best val loss: {best_val_loss:.4f}")
    return model, vocab


if __name__ == "__main__":
    main()
'''
with open(f'{PROJECT}/src/training/train_caption.py', 'w') as f:
    f.write(train_caption_py)
print("train_caption.py written")

## 5. Download MSR-VTT Dataset

MSR-VTT is available on Kaggle. You need to provide your Kaggle API credentials.

**How to get `kaggle.json`:**
1. Go to kaggle.com → Account → API → Create New Token
2. Upload the downloaded `kaggle.json` when the cell below prompts you

**Dataset:** `https://www.kaggle.com/datasets/vishnutheepb/msrvtt`

In [ ]:
import os
from google.colab import files

# Option A: Upload kaggle.json now
print("Upload your kaggle.json file:")
uploaded = files.upload()  # upload kaggle.json

os.makedirs('/root/.config/kaggle', exist_ok=True)
import shutil
shutil.move('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print("Kaggle credentials configured.")

In [ ]:
# Download and extract MSR-VTT
# This downloads ~6 GB — takes ~5–10 min on Colab
!cd {PROJECT} && kaggle datasets download -d vishnutheepb/msrvtt -p data/raw/msvtt --unzip

# List what was downloaded
!ls -lh {PROJECT}/data/raw/msvtt/

In [ ]:
# Verify the expected files exist
import os

anno_file = f'{PROJECT}/data/raw/msvtt/train_val_videodatainfo.json'
video_dir = f'{PROJECT}/data/raw/msvtt/TrainValVideo'

print(f"Annotation file exists : {os.path.exists(anno_file)}")
print(f"Video dir exists       : {os.path.isdir(video_dir)}")

if os.path.isdir(video_dir):
    videos = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    print(f"Videos found           : {len(videos)}")

> **Alternative:** If you already downloaded MSR-VTT and saved it to Google Drive, skip the Kaggle cells and run this instead:
> ```python
> !cp -r /content/drive/MyDrive/msvtt /content/ai-video-summarizer/data/raw/
> ```

## 6. Build Caption Dataset

Extracts ResNet18 features (512-dim) from each video at 2-second intervals and pairs them with captions.

**Expected output:** `data/processed/msvtt/dataset.pkl` (~2–4 GB) + `outputs/models/caption_vocab.pkl`

> This step takes **30–60 minutes** on a T4 GPU for all 7,180 videos.

In [ ]:
import sys
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

# Check if dataset already exists (resume from Drive)
DATASET_PKL = f'{PROJECT}/data/processed/msvtt/dataset.pkl'
VOCAB_PKL = f'{PROJECT}/outputs/models/caption_vocab.pkl'
DRIVE_DATASET = f'{DRIVE_SAVE}/dataset.pkl'
DRIVE_VOCAB = f'{DRIVE_SAVE}/caption_vocab.pkl'

if os.path.exists(DRIVE_DATASET) and not os.path.exists(DATASET_PKL):
    print("Found dataset.pkl in Drive — copying to avoid re-extraction...")
    import shutil
    shutil.copy(DRIVE_DATASET, DATASET_PKL)
    shutil.copy(DRIVE_VOCAB, VOCAB_PKL)
    print("Copied from Drive.")
elif os.path.exists(DATASET_PKL):
    print("dataset.pkl already exists — skipping extraction.")
else:
    print("Building dataset from scratch...")
    from src.data.build_caption_dataset import main as build_dataset
    build_dataset()

In [ ]:
# Back up to Drive immediately after building
import shutil

if os.path.exists(DATASET_PKL) and not os.path.exists(DRIVE_DATASET):
    print("Backing up dataset to Drive...")
    shutil.copy(DATASET_PKL, DRIVE_DATASET)
    shutil.copy(VOCAB_PKL, DRIVE_VOCAB)
    print("Backed up to:", DRIVE_SAVE)
else:
    print("Drive backup already exists or dataset not built yet.")

## 7. Train the Captioning Model

**Architecture:** BiGRU encoder → Bahdanau attention → LSTM decoder  
**Config:** embed=256, encoder_hidden=512, decoder_hidden=512, attention=256, 30 epochs  
**Expected time:** ~2–4 hours on T4 for 30 epochs

In [ ]:
# Verify dataset is ready
import pickle

with open(DATASET_PKL, 'rb') as f:
    ds = pickle.load(f)
with open(VOCAB_PKL, 'rb') as f:
    vocab = pickle.load(f)

print(f"Dataset samples : {len(ds):,}")
print(f"Vocab size      : {len(vocab):,}")
print(f"Sample keys     : {list(ds[0].keys())}")
print(f"Feature shape   : {ds[0]['features'].shape}")
print(f"Caption (encoded): {ds[0]['caption']}")
print(f"Caption (text)  : {ds[0]['caption_text']}")

In [ ]:
# ── Run Training ───────────────────────────────────────────────────────────────
import math
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from src.models.caption_model import VideoCaptionModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {DEVICE}")

# Hyperparameters
EMBED_DIM       = 256
ENCODER_HIDDEN  = 512
DECODER_HIDDEN  = 512
ATTENTION_DIM   = 256
INPUT_DIM       = 512
ENCODER_LAYERS  = 2
DROPOUT         = 0.3
BATCH_SIZE      = 128
EPOCHS          = 30
LR              = 1e-3
TEACHER_FORCING = 0.5
MAX_SEQ_LEN     = 60
VAL_SPLIT       = 0.1
CLIP_GRAD       = 1.0
MODEL_OUT       = f'{PROJECT}/outputs/models/caption_model.pt'


class CaptionDataset(Dataset):
    def __init__(self, samples, max_seq_len=60):
        self.samples = samples
        self.max_seq_len = max_seq_len

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        features = torch.tensor(s['features'], dtype=torch.float32)
        if len(features) > self.max_seq_len:
            features = features[:self.max_seq_len]
        caption = torch.tensor(s['caption'], dtype=torch.long)
        return features, caption, len(features)


def collate_fn(batch):
    features, captions, lengths = zip(*batch)
    max_len = max(lengths)
    feat_dim = features[0].shape[-1]
    padded = torch.zeros(len(features), max_len, feat_dim)
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded[i, :l] = f
    return padded, torch.stack(captions), torch.tensor(lengths, dtype=torch.long)


random.seed(42)
random.shuffle(ds)
split = int(len(ds) * (1 - VAL_SPLIT))
train_set = CaptionDataset(ds[:split], MAX_SEQ_LEN)
val_set   = CaptionDataset(ds[split:], MAX_SEQ_LEN)
print(f"Train: {len(train_set):,}  |  Val: {len(val_set):,}")

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

model = VideoCaptionModel(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    encoder_hidden=ENCODER_HIDDEN,
    decoder_hidden=DECODER_HIDDEN,
    attention_dim=ATTENTION_DIM,
    input_dim=INPUT_DIM,
    encoder_layers=ENCODER_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)
best_val_loss = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    t_loss, t_tokens = 0.0, 0
    for features, captions, _ in train_loader:
        features, captions = features.to(DEVICE), captions.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(features, captions, teacher_forcing_ratio=TEACHER_FORCING)
        out_flat = outputs[:, 1:].reshape(-1, outputs.size(-1))
        tgt_flat = captions[:, 1:].reshape(-1)
        loss = criterion(out_flat, tgt_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()
        non_pad = (tgt_flat != 0).sum().item()
        t_loss += loss.item() * non_pad
        t_tokens += non_pad
    train_loss = t_loss / max(t_tokens, 1)

    # ── Validate ──
    model.eval()
    v_loss, v_tokens = 0.0, 0
    references, hypotheses = [], []
    with torch.no_grad():
        for features, captions, _ in val_loader:
            features, captions = features.to(DEVICE), captions.to(DEVICE)
            outputs = model(features, captions, teacher_forcing_ratio=0.0)
            out_flat = outputs[:, 1:].reshape(-1, outputs.size(-1))
            tgt_flat = captions[:, 1:].reshape(-1)
            loss = criterion(out_flat, tgt_flat)
            non_pad = (tgt_flat != 0).sum().item()
            v_loss += loss.item() * non_pad
            v_tokens += non_pad
            preds = outputs.argmax(dim=-1)
            for i in range(len(captions)):
                references.append([vocab.decode(captions[i].cpu().tolist()).split()])
                hypotheses.append(vocab.decode(preds[i].cpu().tolist()).split())
    val_loss = v_loss / max(v_tokens, 1)
    bleu = corpus_bleu(references, hypotheses)
    scheduler.step(val_loss)

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'bleu': bleu})
    print(
        f"Epoch {epoch:3d}/{EPOCHS} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"PPL {math.exp(val_loss):.1f} | "
        f"BLEU-4 {bleu:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                'epoch': epoch,
                'model_state': model.state_dict(),
                'vocab_size': len(vocab),
                'config': {
                    'embed_dim': EMBED_DIM, 'encoder_hidden': ENCODER_HIDDEN,
                    'decoder_hidden': DECODER_HIDDEN, 'attention_dim': ATTENTION_DIM,
                    'input_dim': INPUT_DIM, 'encoder_layers': ENCODER_LAYERS, 'dropout': DROPOUT,
                },
            },
            MODEL_OUT,
        )
        print(f"           -> best model saved  (val={val_loss:.4f})")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

## 8. Save Results to Google Drive

In [ ]:
import shutil

files_to_save = [
    (MODEL_OUT,  f'{DRIVE_SAVE}/caption_model.pt'),
    (VOCAB_PKL,  f'{DRIVE_SAVE}/caption_vocab.pkl'),
]

for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f"Saved {os.path.basename(dst):30s}  ({size_mb:.1f} MB)  → {dst}")
    else:
        print(f"MISSING: {src}")

print("\nAll outputs saved to:", DRIVE_SAVE)

## 9. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs     = [h['epoch']      for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss']   for h in history]
bleu_scores = [h['bleu']     for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, train_loss, label='Train Loss')
ax1.plot(epochs, val_loss,   label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, bleu_scores, color='green', label='BLEU-4')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('BLEU-4')
ax2.set_title('BLEU-4 Score on Validation Set')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(f'{DRIVE_SAVE}/training_curves.png', dpi=150)
plt.show()
print("Plot saved to Drive.")

## 10. Quick Inference Test

In [ ]:
# Test the model on a few validation samples
model.eval()
sample_batch = next(iter(val_loader))
features, captions, _ = sample_batch
features, captions = features.to(DEVICE), captions.to(DEVICE)

with torch.no_grad():
    outputs = model(features, captions, teacher_forcing_ratio=0.0)
    preds = outputs.argmax(dim=-1)

print("Sample predictions (first 5):")
print('-' * 60)
for i in range(min(5, len(captions))):
    gt  = vocab.decode(captions[i].cpu().tolist())
    hyp = vocab.decode(preds[i].cpu().tolist())
    print(f"Ground truth : {gt}")
    print(f"Prediction   : {hyp}")
    print()